# Vanguard 13F — factor risk, straight from the Atoti cube in Python

This notebook reproduces every view in the **Vanguard 13F filings** folder by calling the Atoti cube
**directly from Python** — `cube.query(...)` in the kernel, no FastAPI, no HTTP, no Streamlit.
Each view below is a self-contained, hand-written query feeding a pandas table (grids) or an
altair chart (charts). The view JSON files were the *spec*; the code here is the literal API.

Run order: build the cube once (next cell), then any view cell. Built on free/public data
(SEC 13F + EDGAR XBRL + Stooq), Vanguard Group's 13F book as a weight overlay.

In [1]:
# The notebook container is air-gapped and its filesystem is read-only, so altair/narwhals
# are staged on the host at data/_pylibs (visible read-only as /app/data/_pylibs).
# No-op outside the container, where altair is installed in the barra/ venv normally.
import os, sys
_libs = "/app/data/_pylibs"
if os.path.isdir(_libs) and _libs not in sys.path:
    sys.path.insert(0, _libs)

import datetime as dt
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()   # whole-market book: sector/day frames exceed altair's 5k default
import notebook_helpers as N

session, cube = N.build()                       # builds the six-frame cube on :9096 (~1-2 min)
h, l, m = cube.hierarchies, cube.levels, cube.measures
D = dt.date(2024, 12, 31)                        # latest monthly COB in the sample; all views as-of D
BOOK = l["Book"] == "Vanguard"
print("cube ready — hierarchies:", sorted(n for _, n in h))

Welcome to Atoti 0.9.15!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.


cube built: 6,004,074 leaf rows, 22 style factors, 130 scenario sets


cube ready — hierarchies: ['Book', 'CorrStress', 'Date', 'FactorDim', 'Manager', 'PositionRank', 'ScenarioDay', 'ScenarioSet', 'Security', 'StressShock']


## Direct cube access — the two idioms

Everything below is built from two `cube.query` shapes: **(1)** scalar measures grouped by a
level under a `filter`, and **(2)** the synthetic **`ScenarioDay`** dimension, which unpacks a
scenario P&L *vector* into one row per day — so a vector measure becomes an ordinary group-by.

In [2]:
# (1) scalar measures, grouped by ScenarioSet, sliced to Vanguard @ latest COB — "the switch":
#     the same measures, every scenario mode, side by side.
display(cube.query(
    m["Scenario VaR 99"], m["Scenario worst loss"], m["Total VaR 99"],
    mode="raw", levels=[l["ScenarioSet"]], filter=BOOK & (l["Date"] == D)))

# (2) ScenarioDay unpacks the COVID P&L vector into a per-day series (head):
display(cube.query(
    m["Scenario PnL at day"], m["Scenario date at day (epoch)"],
    mode="raw", levels=[l["ScenarioDay"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "Evt:COVID2020")).head())

,ScenarioSet,Scenario VaR 99,Scenario worst loss,Total VaR 99
0,Evt:COVID2020,0.108479,0.134881,0.108536
1,Evt:Rates2022,0.041925,0.043934,0.042071
2,Evt:Selloff2018,0.035629,0.036652,0.035801
3,HistFull,0.035102,0.134881,0.035277
4,Hypo:MomentumCrash,0.002019,0.002019,0.004044
...,...,...,...,...
125,PIT:2026-02-28,0.035156,0.134881,0.03533
126,PIT:2026-03-31,0.035145,0.134881,0.035319
127,PIT:2026-04-30,0.035134,0.134881,0.035309
128,PIT:2026-05-31,0.035125,0.134881,0.035299


,ScenarioDay,Scenario PnL at day,Scenario date at day (epoch)
0,0,0.008857,18295
1,1,0.021128,18296
2,2,0.011572,18297
3,3,0.003477,18298
4,4,-0.00947,18299


## L1 · Book risk summary

In [3]:
# ── L1 · Book risk summary ────────────────────────────────────────────────────────────────
# The top of the drill-down: one row, the whole book. Vanguard's 1-day 99% VaR right now, split into
# the factor-driven tail (Scenario VaR 99, historical simulation) and the idiosyncratic tail
# (Specific vol), then combined (Total VaR 99). Everything below decomposes THIS number.
# Direct API: no group-by beyond Book; slice to Vanguard / latest COB / the full-history set.
df = cube.query(
    m["Total VaR 99"], m["Scenario VaR 99"], m["Specific vol"],
    mode="raw", levels=[l["Book"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
)
N.style_grid(df)

,Book,Total VaR 99,Scenario VaR 99,Specific vol
0,Vanguard,3.528%,3.510%,0.151%


## L2 · Factor contributions

In [4]:
# ── L2 · Factor contributions ─────────────────────────────────────────────────────────────
# Drill the book VaR onto the risk factors. Marginal Scenario VaR 99 is each factor's P&L on the
# book's 1%-tail scenario (ADDITIVE — the factors sum to the book factor-VaR); % of Scenario VaR
# 99 is its share. "Which factors own the tail?" — here Market dominates a long-equity book.
# Direct API: group by Factor; same Vanguard / COB / HistFull slice; sorted biggest-first.
df = cube.query(
    m["Marginal Scenario VaR 99"], m["% of Scenario VaR 99"],
    mode="raw", levels=[l["Factor"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Scenario VaR 99", ascending=False)
N.style_grid(df)

,FactorGroup,Factor,Marginal Scenario VaR 99,% of Scenario VaR 99
11,Market,Market,3.267%,93.006%
12,Style,Beta,0.346%,9.860%
7,Industry,Ind:Information Technology,0.153%,4.343%
5,Industry,Ind:Health Care,0.052%,1.472%
6,Industry,Ind:Industrials,0.033%,0.947%
21,Style,Size,0.023%,0.649%
1,Industry,Ind:Consumer Discretionary,0.020%,0.557%
22,Style,Value,0.014%,0.408%
2,Industry,Ind:Consumer Staples,-0.000%,-0.001%
14,Style,Leverage,-0.000%,-0.005%


## L2 · Factor incremental VaR

In [5]:
# ── L2 · Factor incremental VaR ───────────────────────────────────────────────────────────
# The diversification-aware companion to contributions: Incremental Scenario VaR 99 is how much
# book VaR is RELEASED if a factor's exposure is removed (the book tail recomputed without it).
# Unlike the marginals it is NOT additive (VaR is sub-additive) — "what does cutting this factor
# actually buy me?". Direct API: group by Factor; sort by the incremental column.
df = cube.query(
    m["Marginal Scenario VaR 99"], m["Incremental Scenario VaR 99"],
    mode="raw", levels=[l["Factor"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Incremental Scenario VaR 99", ascending=False)
N.style_grid(df)

,FactorGroup,Factor,Marginal Scenario VaR 99,Incremental Scenario VaR 99
11,Market,Market,3.267%,2.618%
12,Style,Beta,0.346%,0.269%
19,Style,RateBeta,-0.012%,0.067%
7,Industry,Ind:Information Technology,0.153%,0.056%
17,Style,NdxBeta,-0.148%,0.040%
5,Industry,Ind:Health Care,0.052%,0.039%
6,Industry,Ind:Industrials,0.033%,0.033%
4,Industry,Ind:Financials,-0.048%,0.022%
3,Industry,Ind:Energy,-0.023%,0.019%
22,Style,Value,0.014%,0.014%


## L2 · Issuer contributions

In [6]:
# ── L2 · Issuer contributions ─────────────────────────────────────────────────────────────
# Same decomposition, now by name. Marginal Total VaR 99 is each issuer's additive share of the
# COMBINED tail (factor + specific in quadrature — the Euler split), meaningful per-name where
# idiosyncratic risk lives. "Which positions carry the book's risk?". Direct API: group by Issuer
# (the cube returns the Country/Sector/Issuer security path); top names first.
df = cube.query(
    m["Marginal Total VaR 99"], m["% of Total VaR 99"],
    mode="raw", levels=[l["Issuer"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Total VaR 99", ascending=False)
N.style_grid(df)

,Country,Sector,Issuer,Marginal Total VaR 99,% of Total VaR 99
2567,US,Information Technology,MICROSOFT CORP,0.253%,7.157%
2390,US,Information Technology,Apple Inc.,0.236%,6.684%
2593,US,Information Technology,NVIDIA CORP,0.231%,6.537%
272,US,Consumer Discretionary,AMAZON COM INC,0.123%,3.498%
2384,US,Information Technology,Alphabet Inc.,0.121%,3.441%
2575,US,Information Technology,"Meta Platforms, Inc.",0.084%,2.388%
2419,US,Information Technology,Broadcom Inc.,0.077%,2.167%
547,US,Consumer Discretionary,"Tesla, Inc.",0.057%,1.623%
1442,US,Health Care,ELI LILLY & Co,0.043%,1.215%
1035,US,Financials,JPMORGAN CHASE & CO,0.035%,0.990%


## L2 · Issuer incremental VaR

In [7]:
# ── L2 · Issuer incremental VaR ───────────────────────────────────────────────────────────
# Per-name diversification view: Incremental Total VaR 99 = book Total VaR released by removing
# the name (factor tail re-struck on the reduced book + its specific variance stripped). The cut
# list a PM reads to de-risk. Direct API: group by Issuer; sort by the incremental column.
df = cube.query(
    m["Marginal Total VaR 99"], m["Incremental Total VaR 99"],
    mode="raw", levels=[l["Issuer"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Incremental Total VaR 99", ascending=False)
N.style_grid(df)

,Country,Sector,Issuer,Marginal Total VaR 99,Incremental Total VaR 99
2593,US,Information Technology,NVIDIA CORP,0.231%,0.332%
2390,US,Information Technology,Apple Inc.,0.236%,0.248%
2567,US,Information Technology,MICROSOFT CORP,0.253%,0.241%
272,US,Consumer Discretionary,AMAZON COM INC,0.123%,0.129%
2384,US,Information Technology,Alphabet Inc.,0.121%,0.121%
2419,US,Information Technology,Broadcom Inc.,0.077%,0.105%
2575,US,Information Technology,"Meta Platforms, Inc.",0.084%,0.098%
547,US,Consumer Discretionary,"Tesla, Inc.",0.057%,0.077%
1442,US,Health Care,ELI LILLY & Co,0.043%,0.043%
1035,US,Financials,JPMORGAN CHASE & CO,0.035%,0.036%


## L3 · FactorGroup contributions

In [8]:
# ── L3 · FactorGroup contributions ────────────────────────────────────────────────────────
# Roll the factor contributions up one level — Market vs Style — for the one-line "is this a
# market bet or a style bet?" read. Same additive Marginal Scenario VaR 99 / share, grouped at
# the FactorGroup level of the FactorDim hierarchy. Direct API: levels=[FactorGroup].
df = cube.query(
    m["Marginal Scenario VaR 99"], m["% of Scenario VaR 99"],
    mode="raw", levels=[l["FactorGroup"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Scenario VaR 99", ascending=False)
N.style_grid(df)

,FactorGroup,Marginal Scenario VaR 99,% of Scenario VaR 99
1,Market,3.267%,93.006%
0,Industry,0.125%,3.547%
2,Style,0.121%,3.447%


## L3 · Sector contributions

In [9]:
# ── L3 · Sector contributions ─────────────────────────────────────────────────────────────
# The book's risk by GICS sector — additive Marginal Total VaR 99 per sector (factor + specific),
# the cross-sectional concentration view. Group at the Sector level of the Security hierarchy (the
# cube returns Country/Sector). Direct API: levels=[Sector]; biggest contributors first.
df = cube.query(
    m["Marginal Total VaR 99"], m["% of Total VaR 99"],
    mode="raw", levels=[l["Sector"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Total VaR 99", ascending=False)
N.style_grid(df)

,Country,Sector,Marginal Total VaR 99,% of Total VaR 99
60,US,Information Technology,1.464%,41.480%
59,US,Industrials,0.532%,15.071%
54,US,Consumer Discretionary,0.443%,12.539%
57,US,Financials,0.294%,8.339%
58,US,Health Care,0.294%,8.335%
62,US,Real Estate,0.118%,3.346%
55,US,Consumer Staples,0.110%,3.124%
64,US,Utilities,0.077%,2.192%
61,US,Materials,0.068%,1.932%
53,US,Communication Services,0.041%,1.169%


## Stress · Factor contributions (COVID 2020)

In [10]:
# ── Stress · Factor contributions (COVID 2020) ────────────────────────────────────────────
# The L2 factor decomposition re-struck under a STRESS set instead of the full history: the ONLY
# change from "L2 · Factor contributions" is the ScenarioSet slice (Evt:COVID2020). That one
# switch — slicing the ScenarioSet hierarchy — turns historical-sim VaR into event-replay stress
# VaR, and Market's tail balloons. Direct API: identical query, ScenarioSet == 'Evt:COVID2020'.
df = cube.query(
    m["Marginal Scenario VaR 99"], m["% of Scenario VaR 99"],
    mode="raw", levels=[l["Factor"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "Evt:COVID2020"),
).sort_values("Marginal Scenario VaR 99", ascending=False)
N.style_grid(df)

,FactorGroup,Factor,Marginal Scenario VaR 99,% of Scenario VaR 99
11,Market,Market,12.742%,94.468%
12,Style,Beta,0.652%,4.831%
21,Style,Size,0.333%,2.467%
9,Industry,Ind:Real Estate,0.316%,2.341%
1,Industry,Ind:Consumer Discretionary,0.230%,1.703%
18,Style,NonLinSize,0.167%,1.242%
16,Style,Momentum,0.075%,0.559%
4,Industry,Ind:Financials,0.071%,0.528%
10,Industry,Ind:Utilities,0.051%,0.376%
13,Style,EarnYield,0.013%,0.094%


## VaR trend (HistFull)

In [11]:
# ── VaR trend (HistFull) ──────────────────────────────────────────────────────────────────
# The book's risk THROUGH TIME: 99% factor VaR, total VaR, and specific vol at every monthly COB
# (the full 2016-2024 calendar, historical-sim set). One line per measure — "is the book getting
# riskier?". Direct API: group by Date (no Date filter -> the whole series); melt the three
# measure columns to long form for a colour-per-measure line in altair.
trend = cube.query(
    m["Scenario VaR 99"], m["Total VaR 99"], m["Specific vol"],
    mode="raw", levels=[l["Date"]],
    filter=BOOK & (l["ScenarioSet"] == "HistFull"),
).reset_index().sort_values("Date")
meas = ["Scenario VaR 99", "Total VaR 99", "Specific vol"]
trend["Date"] = pd.to_datetime(trend["Date"])
trend[meas] = trend[meas].astype(float)
long = trend.melt(id_vars="Date", value_vars=meas, var_name="Measure", value_name="Value")

alt.Chart(long).mark_line(point=True).encode(
    x=alt.X("Date:T", title=None),
    y=alt.Y("Value:Q", axis=alt.Axis(format="%"), title="% of NAV"),
    color=alt.Color("Measure:N", scale=alt.Scale(domain=meas,
        range=["#2a4d69", "#c46d4e", "#6b8f71"]), legend=alt.Legend(orient="top", title=None)),
    tooltip=["Date:T", "Measure:N", alt.Tooltip("Value:Q", format=".4f")],
).properties(height=380, width="container", title="VaR trend — Vanguard (HistFull)")

alt.Chart(...)

## Stress board — all scenarios

In [12]:
# ── Stress board · all scenarios ──────────────────────────────────────────────────────────
# Every scenario set side by side at the latest COB: factor VaR, worst single-day loss, and total
# VaR per set — historical-sim vs the three event replays vs the hypotheticals. The CRO's "how bad
# across regimes" board. Direct API: group by ScenarioSet (no ScenarioSet filter -> all sets);
# melt to long for grouped horizontal bars, worst-first.
board = cube.query(
    m["Scenario VaR 99"], m["Scenario worst loss"], m["Total VaR 99"],
    mode="raw", levels=[l["ScenarioSet"]],
    filter=BOOK & (l["Date"] == D),
).reset_index()
meas = ["Scenario VaR 99", "Scenario worst loss", "Total VaR 99"]
board[meas] = board[meas].astype(float)
long = board.melt(id_vars="ScenarioSet", value_vars=meas, var_name="Measure", value_name="Value")

alt.Chart(long).mark_bar().encode(
    y=alt.Y("ScenarioSet:N", title=None, sort="-x"),
    x=alt.X("Value:Q", axis=alt.Axis(format="%"), title="% of NAV"),
    yOffset=alt.YOffset("Measure:N"),
    color=alt.Color("Measure:N", scale=alt.Scale(domain=meas,
        range=["#2a4d69", "#c46d4e", "#6b8f71"]), legend=alt.Legend(orient="top", title=None)),
    tooltip=["ScenarioSet:N", "Measure:N", alt.Tooltip("Value:Q", format=".4f")],
).properties(height=420, width="container", title="Stress board — Vanguard @ 2024-12-31")

alt.Chart(...)

## Tail risk — VaR ladder & Expected Shortfall

The full tail across every scenario set: VaR at 95 / 97.5 / 99, then **Expected Shortfall** (ES /
CVaR) at 97.5 / 99 — the *mean* loss beyond VaR and the Basel FRTB measure that replaced VaR.

In [13]:
# ── Tail risk · VaR ladder & Expected Shortfall ────────────────────────────────
# The full tail, every scenario set side by side: VaR at 95 / 97.5 / 99 (the ladder reads how fat
# the tail is), then Expected Shortfall (ES, a.k.a. CVaR) at 97.5 / 99 — the MEAN loss beyond VaR,
# the Basel FRTB measure that replaced VaR (ES97.5 is the regulatory point). Scenario PnL vol is the
# per-observation dispersion (null for the length-1 hypotheticals). ES >= VaR at the same level by
# construction; ES97.5 ~ VaR99 under a normal tail. Direct API: group by ScenarioSet, all sets.
ladder = cube.query(
    m["Scenario VaR 95"], m["Scenario VaR 97.5"], m["Scenario VaR 99"],
    m["Scenario ES 97.5"], m["Scenario ES 99"], m["Scenario PnL vol"],
    mode="raw", levels=[l["ScenarioSet"]],
    filter=BOOK & (l["Date"] == D),
).sort_values("Scenario ES 97.5", ascending=False)
N.style_grid(ladder)

,ScenarioSet,Scenario VaR 95,Scenario VaR 97.5,Scenario VaR 99,Scenario ES 97.5,Scenario ES 99,Scenario PnL vol
0,Evt:COVID2020,5.345%,9.002%,10.848%,10.927%,13.488%,3.848%
57,PIT:2020-06-30,1.958%,2.808%,3.896%,4.648%,6.450%,1.385%
59,PIT:2020-08-31,1.877%,2.743%,3.784%,4.584%,6.450%,1.368%
58,PIT:2020-07-31,1.921%,2.781%,3.839%,4.584%,6.450%,1.378%
61,PIT:2020-10-31,1.965%,2.830%,3.673%,4.573%,6.450%,1.372%
60,PIT:2020-09-30,1.958%,2.815%,3.730%,4.561%,6.450%,1.374%
55,PIT:2020-04-30,1.805%,2.729%,3.746%,4.520%,6.422%,1.352%
63,PIT:2020-12-31,1.905%,2.805%,3.649%,4.517%,6.235%,1.363%
62,PIT:2020-11-30,1.958%,2.814%,3.658%,4.517%,6.235%,1.372%
56,PIT:2020-05-31,1.886%,2.731%,3.694%,4.487%,6.422%,1.364%


## L2 · Factor ES contributions

The ES analogue of the factor VaR decomposition: each factor's mean P&L over the book's worst-k
tail scenarios. Additive — the factors sum to the book **Scenario ES 97.5**.

In [14]:
# ── L2 · Factor ES contributions ─────────────────────────────────────
# The ES analogue of "L2 · Factor contributions": each factor's MEAN P&L over the book's worst-k
# tail scenarios (k = 2.5% of the history). Additive — the factors sum to the book Scenario ES 97.5
# — and % of Scenario ES 97.5 is the share. Because ES averages the whole tail (not the single 1%
# day) it is the steadier "which factors own the tail?" read. Direct API: group by Factor, HistFull.
df = cube.query(
    m["Marginal Scenario ES 97.5"], m["% of Scenario ES 97.5"],
    mode="raw", levels=[l["Factor"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Scenario ES 97.5", ascending=False)
N.style_grid(df)

,FactorGroup,Factor,Marginal Scenario ES 97.5,% of Scenario ES 97.5
11,Market,Market,3.501%,91.426%
12,Style,Beta,0.313%,8.161%
17,Style,NdxBeta,0.027%,0.712%
18,Style,NonLinSize,0.015%,0.395%
6,Industry,Ind:Industrials,0.013%,0.330%
9,Industry,Ind:Real Estate,0.011%,0.295%
3,Industry,Ind:Energy,0.007%,0.178%
20,Style,ResidVol,0.007%,0.177%
16,Style,Momentum,0.005%,0.130%
19,Style,RateBeta,0.005%,0.128%


## Concentration — Risk HHI

Single-number name concentration: the Herfindahl index of each name's share of book Total VaR.
1/N for an evenly-diversified book up to 1.0 for a single name — the limit-monitoring gauge.

In [15]:
# ── Concentration · Risk HHI ─────────────────────────────────────────
# Single-number name concentration: the Herfindahl index of each name's share of book Total VaR,
# the sum of share^2 over names. 1/N for an evenly-diversified book (here ~0.03 under the full
# history, ~30-odd effective names) up to 1.0 for a single name; the single-factor hypotheticals
# read far higher because the shock concentrates risk in a handful of loaded names. as_pct off —
# HHI is an index, not a percent. Direct API: group by ScenarioSet.
# NB (124-book build, 2026-08-14): on a whole-market book this size (~3.6k names) the
# all-sets Risk HHI query materializes per-name risk shares for every scenario set at
# once and exhausts the notebook cube heap — sliced to HistFull, same as the /trends
# date-loop precedent. (Risk HHI is legacy on the desk anyway — Top-5 risk share
# replaced it in the limits/monitor views.)
hhi = cube.query(
    m["Risk HHI"],
    mode="raw", levels=[l["ScenarioSet"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Risk HHI", ascending=False)
N.style_grid(hhi, pct=False)

,ScenarioSet,Risk HHI
0,HistFull,0.019


## Scenario P&L — COVID 2020

Two structurally-**different** graphs, built the explicit way: **two `cube.query` calls → two
DataFrames → two graphs**, one graph per DataFrame. The query (a tabular dataset) is the primitive;
the graph just references it. The scenario engine produces a P&L *vector* over the COVID replay
window; the synthetic `ScenarioDay` dimension unpacks it. **Query 1** is the book path
(`levels=[ScenarioDay]`); **Query 2** adds a grain (`levels=[ScenarioDay, Sector]`) and is
re-ordered worst→best into a loss curve — the ordering computed **in the DataFrame**, not the chart.

In [16]:
# ── COVID 2020 · QUERY 1 of 2 -> the book P&L path DataFrame ───────────────────────────────
# Two different graphs need two different queries -> two DataFrames. This is query 1: the per-day
# BOOK path. ScenarioDay unpacks the scenario P&L vector into one row per day WITHOUT exploding the
# cube, so the path is a plain group-by: levels=[ScenarioDay]. The 99% VaR threshold and the worst-
# loss day ride along as book-level marker measures. Epoch-day ints -> real dates in pandas.
covid = BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "Evt:COVID2020")
covid_path = cube.query(
    m["Scenario PnL at day"], m["Scenario date at day (epoch)"],
    m["Scenario VaR line at day"], m["Scenario worst pnl at day"],
    m["Scenario worst date at day (epoch)"],
    mode="raw", levels=[l["ScenarioDay"]], filter=covid,
).reset_index()
for c in ["Scenario PnL at day", "Scenario VaR line at day", "Scenario worst pnl at day"]:
    covid_path[c] = covid_path[c].astype(float)
covid_path["date"]       = pd.to_datetime(covid_path["Scenario date at day (epoch)"].astype("int64"), unit="D")
covid_path["worst_date"] = pd.to_datetime(covid_path["Scenario worst date at day (epoch)"].astype("int64"), unit="D")
covid_path        # DataFrame 1 — the tabular dataset graph 1 draws from

,index,ScenarioDay,Scenario PnL at day,Scenario date at day (epoch),Scenario VaR line at day,Scenario worst pnl at day,Scenario worst date at day (epoch),date,worst_date
0,0,0,0.008857,18295,-0.108479,-0.134881,18337,2020-02-03,2020-03-16
1,1,1,0.021128,18296,-0.108479,-0.134881,18337,2020-02-04,2020-03-16
2,2,2,0.011572,18297,-0.108479,-0.134881,18337,2020-02-05,2020-03-16
3,3,3,0.003477,18298,-0.108479,-0.134881,18337,2020-02-06,2020-03-16
4,4,4,-0.009470,18299,-0.108479,-0.134881,18337,2020-02-07,2020-03-16
...,...,...,...,...,...,...,...,...,...
77,77,77,0.001869,18404,-0.108479,-0.134881,18337,2020-05-22,2020-03-16
78,78,78,0.019108,18408,-0.108479,-0.134881,18337,2020-05-26,2020-03-16
79,79,79,0.020047,18409,-0.108479,-0.134881,18337,2020-05-27,2020-03-16
80,80,80,-0.008181,18410,-0.108479,-0.134881,18337,2020-05-28,2020-03-16


In [17]:
# ── COVID 2020 · GRAPH 1 — book scenario P&L path (draws from `covid_path`) ────────────────
# The graph just references DataFrame 1: a P&L line + zero baseline + the red 99% VaR rule and the
# red worst-loss point (book-level markers, collapsed to one mark each with aggregate:min).
base     = alt.Chart(covid_path)
zero     = base.mark_rule(color="#d8d5cd").encode(y=alt.datum(0))
line     = base.mark_line(point=True, color="#2a4d69", strokeWidth=1.3).encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("Scenario PnL at day:Q", axis=alt.Axis(format="%"), title="scenario P&L"),
    tooltip=[alt.Tooltip("date:T"),
             alt.Tooltip("Scenario PnL at day:Q", format=".2%", title="P&L")])
var_rule = base.mark_rule(color="#c0392b", strokeDash=[4, 4]).encode(
    y=alt.Y("min(Scenario VaR line at day):Q"))                      # one book VaR-99 rule
worst    = base.mark_point(color="#c0392b", size=80, filled=True).encode(
    x=alt.X("min(worst_date):T"), y=alt.Y("min(Scenario worst pnl at day):Q"),
    tooltip=[alt.Tooltip("min(worst_date):T", title="worst date"),
             alt.Tooltip("min(Scenario worst pnl at day):Q", format=".2%", title="worst P&L")])

(zero + line + var_rule + worst).properties(
    height=300, width="container", title="COVID 2020 — book scenario P&L path")

alt.LayerChart(...)

In [18]:
# ── COVID 2020 · QUERY 2 of 2 -> the by-Sector loss-curve DataFrame ────────────────────────
# Query 2: the SAME per-day P&L at a finer grain -- levels=[ScenarioDay, Sector] -- so each day
# splits into its sector contributions. The worst->best ordering is computed HERE, in the DataFrame:
# roll the sectors back up to a book total per day, sort those day-totals ascending (worst/most-
# negative first), carry the `rank` onto every sector row, then physically sort the frame. (Graph 2
# then uses sort=None to honour this order.) Tick labels stay dates (%d %b); the order is the rank.
covid_sector = cube.query(
    m["Scenario PnL at day"], m["Scenario date at day (epoch)"],
    mode="raw", levels=[l["ScenarioDay"], l["Sector"]], filter=covid,
).reset_index()
covid_sector["Scenario PnL at day"] = covid_sector["Scenario PnL at day"].astype(float)
covid_sector["date"] = pd.to_datetime(covid_sector["Scenario date at day (epoch)"].astype("int64"), unit="D")

order = (covid_sector.groupby("ScenarioDay", as_index=False)["Scenario PnL at day"]
                     .sum().sort_values("Scenario PnL at day"))       # worst (most negative) first
order["rank"] = range(len(order))
covid_sector = (covid_sector.merge(order[["ScenarioDay", "rank"]], on="ScenarioDay")
                            .sort_values(["rank", "Sector"]))         # df now in worst->best order
covid_sector["label"] = covid_sector["date"].dt.strftime("%d %b")
covid_sector      # DataFrame 2 — the tabular dataset graph 2 draws from

,index,ScenarioDay,Country,Sector,Scenario PnL at day,Scenario date at day (epoch),date,rank,label
2060,2060,29,Brazil,Communication Services,-3.149371e-07,18337,2020-03-16,0,16 Mar
2067,2067,29,Canada,Communication Services,-6.486522e-06,18337,2020-03-16,0,16 Mar
2078,2078,29,Cayman Islands,Communication Services,-8.642578e-06,18337,2020-03-16,0,16 Mar
2094,2094,29,Indonesia,Communication Services,-4.361324e-08,18337,2020-03-16,0,16 Mar
2102,2102,29,Mexico,Communication Services,-6.383584e-08,18337,2020-03-16,0,16 Mar
...,...,...,...,...,...,...,...,...,...
2492,2492,35,Brazil,Utilities,9.038212e-07,18345,2020-03-24,81,24 Mar
2503,2503,35,Canada,Utilities,2.000827e-04,18345,2020-03-24,81,24 Mar
2515,2515,35,Chile,Utilities,3.880916e-08,18345,2020-03-24,81,24 Mar
2526,2526,35,"Korea, Republic of",Utilities,2.127757e-07,18345,2020-03-24,81,24 Mar


In [19]:
# ── COVID 2020 · GRAPH 2 — scenario P&L by Sector, the loss curve (draws from `covid_sector`) ─
# The graph just references DataFrame 2: sectors stacked to the book total, x in the DataFrame's own
# worst->best row order (sort=None), labelled by date.
alt.Chart(covid_sector).mark_area(opacity=0.85, line={"strokeWidth": 0.4}).encode(
    x=alt.X("label:N", sort=None, title="scenario date (worst → best)",   # sort=None => keep df order
            axis=alt.Axis(labelAngle=-45, labelOverlap=True)),
    y=alt.Y("Scenario PnL at day:Q", stack="zero", axis=alt.Axis(format="%"), title="scenario P&L"),
    color=alt.Color("Sector:N", scale=alt.Scale(scheme="set2"),
                    legend=alt.Legend(orient="top", title=None)),
    tooltip=[alt.Tooltip("date:T", title="date"), "Sector:N",
             alt.Tooltip("Scenario PnL at day:Q", format=".2%", title="P&L")],
).properties(height=300, width="container", title="COVID 2020 — scenario P&L by Sector")

alt.Chart(...)